In [ ]:
import json
from pathlib import Path
from typing import Any

import torch

from src.interpretability_pipeline import PipelineConfig, run_batch, run_pipeline

PROJECT_ROOT = Path.cwd()
QUESTIONS_ROOT = PROJECT_ROOT / "questions"
OUTPUT_ROOT = PROJECT_ROOT / "results"
DEVICE: str | None = "cuda" if torch.cuda.is_available() else None

DEVICE


In [ ]:
def safe_name(value: str, max_length: int = 90) -> str:
    cleaned = "".join(char if char.isalnum() or char in "._-" else "_" for char in value)
    cleaned = "_".join(part for part in cleaned.split("_") if part)
    return cleaned[:max_length] or "item"


def load_named_texts(root: Path) -> list[tuple[str, str]]:
    return [
        (path.name, path.read_text(encoding="utf-8").replace("\\n", " ").strip())
        for path in sorted(root.glob("*.txt"))
        if path.is_file()
    ]


english_texts: list[tuple[str, str]] = load_named_texts(QUESTIONS_ROOT / "eng")
polish_texts: list[tuple[str, str]] = load_named_texts(QUESTIONS_ROOT / "pl")
all_texts: list[tuple[str, str, str]] = [
    *(("en", name, text) for name, text in english_texts),
    *(("pl", name, text) for name, text in polish_texts),
]

[(language, name, len(text.split())) for language, name, text in all_texts]


In [ ]:
sample_language, sample_name, sample_text = all_texts[0]
sample_output_dir = OUTPUT_ROOT / "interpretability_demo" / safe_name(sample_name)

sample_config = PipelineConfig(
    pipelines_to_run=("masking", "attention_viz"),
    output_dir=str(sample_output_dir),
    device=DEVICE,
    bert_model_name="bert-base-multilingual-cased",
    attention_scope="summary",
    save_outputs=True,
    viz_output="html",
)

sample_result = run_pipeline(sample_text, sample_config)
sample_result.keys()


In [ ]:
qa_data_candidates = sorted((QUESTIONS_ROOT / "q&a").glob("common_knowledge*kopia.json"))
if not qa_data_candidates:
    raise FileNotFoundError("Expected common_knowledge kopia JSON under questions/q&a")
qa_data_path = qa_data_candidates[0]
with qa_data_path.open("r", encoding="utf-8") as handle:
    qa_data: list[dict[str, Any]] = json.load(handle)

qa_context_texts: list[tuple[str, str, str]] = [
    (
        str(item.get("lang", "qa")),
        f"qa_context_{safe_name(str(item.get('id') or item.get('answer') or index))}.txt",
        str(item["passage"]),
    )
    for index, item in enumerate(qa_data)
]

{
    "qa_items": len(qa_data),
    "qa_context_examples": [(language, name, len(text.split())) for language, name, text in qa_context_texts[:5]],
}


In [ ]:
bert_masking_root = OUTPUT_ROOT / "bert_masking"
qa_context_masking_root = OUTPUT_ROOT / "bert_masking_qa_context"
bert_masking_root.mkdir(parents=True, exist_ok=True)
qa_context_masking_root.mkdir(parents=True, exist_ok=True)

bert_masking_jobs: list[dict[str, object]] = [
    {
        "kind": "text_file",
        "language": language,
        "file_name": file_name,
        "text": text,
        "output_root": bert_masking_root,
    }
    for language, file_name, text in all_texts
]
bert_masking_jobs.extend(
    {
        "kind": "qa_context",
        "language": language,
        "file_name": file_name,
        "text": context_text,
        "output_root": qa_context_masking_root,
    }
    for language, file_name, context_text in qa_context_texts
)

bert_masking_results: list[dict[str, object]] = []
for job in bert_masking_jobs:
    result_dir = Path(job["output_root"]) / str(job["file_name"])
    result_dir.mkdir(parents=True, exist_ok=True)

    config = PipelineConfig(
        pipelines_to_run=("masking",),
        output_dir=str(result_dir),
        device=DEVICE,
        bert_model_name="bert-base-multilingual-cased",
        normalize_scores=False,
        save_outputs=True,
        viz_output="none",
    )
    result = run_pipeline(str(job["text"]), config)
    masking = dict(result["masking"])
    bert_masking_results.append(
        {
            "kind": job["kind"],
            "language": job["language"],
            "file_name": job["file_name"],
            "token_count": len(masking["tokens"]),
            "max_importance": max(float(value) for value in masking["importance"]),
        }
    )

bert_masking_results


In [ ]:
nli_root = OUTPUT_ROOT / "nli"
qa_context_nli_root = OUTPUT_ROOT / "nli_qa_context"
nli_root.mkdir(parents=True, exist_ok=True)
qa_context_nli_root.mkdir(parents=True, exist_ok=True)

nli_jobs: list[dict[str, object]] = [
    {
        "kind": "text_file",
        "language": language,
        "file_name": file_name,
        "text": text,
        "output_root": nli_root,
    }
    for language, file_name, text in all_texts
]
nli_jobs.extend(
    {
        "kind": "qa_context",
        "language": language,
        "file_name": file_name,
        "text": context_text,
        "output_root": qa_context_nli_root,
    }
    for language, file_name, context_text in qa_context_texts
)

nli_results: list[dict[str, object]] = []
for job in nli_jobs:
    language = str(job["language"])
    result_dir = Path(job["output_root"]) / str(job["file_name"])
    result_dir.mkdir(parents=True, exist_ok=True)

    hypothesis_template = "This text is {label}." if language == "en" else "Ten tekst jest {label}."
    hypothesis_label = "complex" if language == "en" else "skomplikowany"
    config = PipelineConfig(
        pipelines_to_run=("decision",),
        output_dir=str(result_dir),
        device=DEVICE,
        nli_model_name="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
        hypothesis_template=hypothesis_template,
        nli_hypothesis_label=hypothesis_label,
        save_outputs=True,
        viz_output="html",
    )
    result = run_pipeline(str(job["text"]), config)
    decision = dict(result["decision"])
    nli_results.append(
        {
            "kind": job["kind"],
            "language": language,
            "file_name": job["file_name"],
            "model_name": decision.get("model_name"),
            "hypothesis": decision.get("hypothesis"),
            "pred_label": decision.get("pred_label"),
            "target_label": decision.get("target_label"),
        }
    )

nli_results


In [ ]:
surprisal_jobs: list[dict[str, object]] = [
    {
        "kind": "text_file",
        "language": language,
        "file_name": file_name,
        "text": text,
        "output_root": OUTPUT_ROOT / ("surprisal" if language == "en" else "surprisal_pl"),
    }
    for language, file_name, text in all_texts
]
surprisal_jobs.extend(
    {
        "kind": "qa_context",
        "language": language,
        "file_name": file_name,
        "text": context_text,
        "output_root": OUTPUT_ROOT / (
            "surprisal_qa_context" if language == "en" else "surprisal_pl_qa_context"
        ),
    }
    for language, file_name, context_text in qa_context_texts
)

surprisal_results: list[dict[str, object]] = []
for job in surprisal_jobs:
    language = str(job["language"])
    result_dir = Path(job["output_root"]) / str(job["file_name"])
    result_dir.mkdir(parents=True, exist_ok=True)

    model_name = "gpt2" if language == "en" else "radlab/polish-gpt2-small-v2"
    config = PipelineConfig(
        pipelines_to_run=("surprisal",),
        output_dir=str(result_dir),
        device=DEVICE,
        surprisal_model=model_name,
        save_outputs=True,
    )
    result = run_pipeline(str(job["text"]), config)
    surprisal = dict(result["surprisal"])
    summary = dict(surprisal.get("summary", {}))
    surprisal_results.append(
        {
            "kind": job["kind"],
            "language": job["language"],
            "file_name": job["file_name"],
            "model": surprisal.get("model_name"),
            "mean_surprisal_bits": summary.get("mean_bits"),
            "max_surprisal_bits": summary.get("max_bits"),
        }
    )

surprisal_results


In [ ]:
qa_root = OUTPUT_ROOT / "qa"
qa_root.mkdir(parents=True, exist_ok=True)

qa_jobs: list[dict[str, object]] = [
    {
        "kind": "qa_context",
        "language": str(item.get("lang", "qa")),
        "file_name": f"qa_context_{safe_name(str(item.get('id') or item.get('answer') or index))}.txt",
        "text": str(item["passage"]),
        "question": str(item["question"]),
        "expected_answer": str(item.get("answer", "")),
        "output_root": qa_root,
    }
    for index, item in enumerate(qa_data)
]

qa_results: list[dict[str, object]] = []
for job in qa_jobs:
    result_dir = Path(job["output_root"]) / str(job["file_name"])
    result_dir.mkdir(parents=True, exist_ok=True)

    config = PipelineConfig(
        pipelines_to_run=("qa",),
        output_dir=str(result_dir),
        device=DEVICE,
        qa_methods=("lig", "feature_ablation"),
        qa_target="both",
        save_outputs=True,
        viz_output="html",
    )
    result = run_pipeline(
        str(job["text"]),
        config,
        question=str(job["question"]),
        context=str(job["text"]),
    )
    qa = dict(result["qa"])
    qa_results.append(
        {
            "kind": job["kind"],
            "language": job["language"],
            "file_name": job["file_name"],
            "expected_answer": job["expected_answer"],
            "predicted_answer": qa.get("predicted_answer"),
            "span_score": qa.get("span_score"),
        }
    )

qa_results


In [ ]:
batch_texts: list[str] = [text for _, _, text in all_texts[:2]]
batch_config = PipelineConfig(
    pipelines_to_run=("masking",),
    output_dir=str(OUTPUT_ROOT / "interpretability_demo_batch"),
    device=DEVICE,
    bert_model_name="bert-base-multilingual-cased",
    save_outputs=True,
    viz_output="none",
)

batch_results = run_batch(batch_texts, batch_config)
[len(result["masking"]["tokens"]) for result in batch_results]
